# Module 3.1: Self-Attention

We finally have all the ingredients: we've turned words into embeddings (Module 2.1),
and we've mathematically injected their position in the sentence (Module 2.2).

Now we build the engine of the Transformer: **the attention mechanism**. This is where
the model learns context by letting every word physically "look" at every other word.

Attention has three moving parts, and cramming them into one sitting is how people end
up copying code they can't debug. So we split them:

| Module | What you build |
|---|---|
| **3.1 (this one)** | **one** attention head: where Q, K, V come from, and the formula end to end |
| 3.2 | **many** heads in parallel, plus the causal mask that stops the model cheating |

By the end of this notebook you will have written a working `SingleHeadAttention` layer
and *seen* the word "bank" decide it belongs with "river".

## 1. The Core Idea: Learnable Q, K, V

### The Concept
In Module 1.2, we learned the math for $Attention(Q, K, V) = softmax(Q K^T / \sqrt{d_k})V$. 

But where do $Q$ (Query), $K$ (Key), and $V$ (Value) come from? We don't just use the raw embeddings. Instead, we use Neural Network **Linear Layers** (Weight Matrices) to mathematically *project* the embedding into three different versions of itself.

### Why do we need it? (Learnable Parameters)
If we just used the raw embeddings, the Attention formula would always output the exact same similarities (e.g. "Apple" would always attend to "Fruit"). By using `Linear` layers, we give the model "knobs" it can turn. During training, the gradients adjust these Linear layers so the model **learns** *what* it should be querying for depending on the task (e.g., in a translation task, it might learn to query for verbs).

## 2. Self-Attention (Single Head)

### The Concept
"Self-Attention" means the sequence is attending to *itself*. Every word in the sentence asks every other word: "Are you relevant to my meaning?"

For example, in the sentence "The bank of the river", the word `bank` will output a Query. The word `river` will have a Key that matches that Query well. They will have a high dot product, and `bank` will absorb the Value vector from `river`—nudging its own representation toward the "nature" sense rather than the "finance" sense.

> Note: with **random, untrained** weights this disambiguation does *not* happen — the projections are meaningless until trained. In the demo below we **hand-craft** the embeddings and projections so you can actually see `bank` attend to `river`.

### Why do we need it? (Parallel Context)
This replaces the old Recurrent Neural Networks (RNNs) which had to read left-to-right, one word at a time. Self-Attention evaluates the entire sentence synchronously using a single large Matrix Multiplication. This lets any word directly reference **any other word in the context window**—no "forgetting" of earlier words—and it runs fast on a GPU. The window is **finite**, though, and because every word compares against every other word, the compute and memory cost grows with the sequence length **squared** ($O(n^2)$). That quadratic cost is the main reason context windows can't simply be made infinite.

### Reading the formula, end to end

The whole mechanism is one line — the single most important equation in the course:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^{T}}{\sqrt{d_k}}\right) V$$

Don't be intimidated — read it in **four stages**, each one a piece you already built:

| Stage | Piece | What it does | Built in |
|:---:|:---:|:---|:---:|
| 1 | $Q K^{T}$ | every query · every key → a grid of raw **similarity scores** | 1.1 (dot product) |
| 2 | $\div \sqrt{d_k}$ | shrink the scores so softmax stays responsive | 1.2 (√dₖ trick) |
| 3 | $\text{softmax}(\cdot)$ | turn each row of scores into **weights that sum to 1** | 1.2 (softmax) |
| 4 | $(\cdots)\,V$ | take a **weighted average of the value vectors** — an *expectation* | 1.1 (expectation) |

And the three inputs:

- $Q$ (**query**) — "what each token is looking for."
- $K$ (**key**) — "what each token offers, to be matched against."
- $V$ (**value**) — "the information a token hands over when it's attended to."
- $K^{T}$ — the **transpose** (Module 1.2) that lets queries dock against keys.
- $d_k$ — the head dimension: how many features each query/key has.

As a picture, the data flows like this:

```mermaid
flowchart LR
    x["x (token vectors)"] --> Q["Q = x Wq"]
    x --> K["K = x Wk"]
    x --> V["V = x Wv"]
    Q --> S["scores = Q Kt"]
    K --> S
    S --> Sc["divide by sqrt(dk)"]
    Sc --> SM["softmax (each row sums to 1)"]
    SM --> O["weighted sum of V"]
    V --> O
    O --> out["output (same shape as x)"]
```

The formula literally *is* those four stages in a row. Let's run them by hand on two
tiny tokens and print every intermediate, so nothing is hidden inside a function:

In [ ]:
import torch
from math import sqrt

# Two tokens, 4-dim, chosen so we can read every number.
Q = torch.tensor([[1., 0., 1., 0.],     # token 0's query
                  [0., 1., 0., 1.]])    # token 1's query
K = torch.tensor([[1., 0., 1., 0.],     # token 0's key (matches token 0's query)
                  [0., 1., 0., 1.]])    # token 1's key
V = torch.tensor([[10.,  0.],           # token 0's value
                  [ 0., 10.]])          # token 1's value
d_k = Q.size(-1)

scores = Q @ K.T                         # STAGE 1: every query . every key
print("1) Q Kᵀ  (raw similarity scores):\n", scores)

scaled = scores / sqrt(d_k)              # STAGE 2: shrink (divide by sqrt(d_k)=2)
print("\n2) scaled by 1/√d_k:\n", scaled)

weights = torch.softmax(scaled, dim=-1)  # STAGE 3: each row -> weights summing to 1
print("\n3) softmax -> attention weights (rows sum to 1):\n", weights)

output = weights @ V                     # STAGE 4: weighted average of the values
print("\n4) weights @ V  (the output = a weighted average / expectation):\n", output)
print("\nToken 0 matched itself, so its output leans to V[0]=[10,0]; token 1 to V[1]=[0,10].")
print("Those four lines ARE the attention formula. Everything else just adds heads and masks.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from math import sqrt

# Reproducibility: identical random weights on every run.
torch.manual_seed(0)

def scaled_dot_product_attention(query, key, value, mask=None):
    dk = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / sqrt(dk)
    if mask is not None:
        # Where the mask is 0 ("not allowed to look here"), set the score to a huge
        # negative number. After softmax, exp(-1e9) is effectively 0, so those
        # positions receive ~zero attention weight. -1e9 is just "large negative".
        scores = scores.masked_fill(mask == 0, -1e9)
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, value)
    return output, attention_weights

class SingleHeadAttention(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        # These are the Learnable Weight Matrices!
        # bias=False: a Query/Key/Value projection only needs to ROTATE/SCALE the
        # embedding direction; a constant bias added to every token would not help
        # distinguish tokens and just wastes parameters. This matches real LLMs.
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        # x shape: (Batch, Seq_Len, d_model)
        Q = self.W_q(x)  # "What am I looking for?"
        K = self.W_k(x)  # "What do I have?"
        V = self.W_v(x)  # "What information will I give you?"

        # Use our math function from Module 1!
        # Output shape is the EXACT same as input shape (Batch, Seq_Len, d_model).
        # That matters later: because the shape is preserved, the attention output
        # can be ADDED back to the input as a residual connection (x + attention(x)).
        output, weights = scaled_dot_product_attention(Q, K, V, mask=mask)
        return output, weights

# 1 Batch, 5 Words, 128 Dimensional Embedding
dummy_sentence_embeddings = torch.randn(1, 5, 128)
single_head = SingleHeadAttention(d_model=128)

contextualized_output, _ = single_head(dummy_sentence_embeddings)
print(f"Input Shape : {dummy_sentence_embeddings.shape}")
print(f"Output Shape: {contextualized_output.shape}")
print("Same shape in and out -> can be used as a residual: x + attention(x)")

### Seeing it work: "bank" attends to "river"

The run above used random weights, so its attention pattern is meaningless. To actually *see* `bank` look at `river`, we **hand-craft** tiny 4-D embeddings where words that belong together point in similar directions, and we use identity Q/K/V projections (i.e. we pretend training already happened). Then we print the attention-weight matrix and a heatmap.

Sentence: `["the", "bank", "of", "the", "river"]`. We design "bank" and "river" to share a "nature" component so their dot product is high.

In [ ]:
import matplotlib.pyplot as plt

words = ["the", "bank", "of", "the", "river"]

# Hand-crafted 4-D embeddings. Dimensions (loosely):
# [function_word, finance, nature, water]
# "bank" and "river" both carry a strong "nature/water" component, so they match.
emb = torch.tensor([
    [1.0, 0.0, 0.0, 0.0],   # the   (function word)
    [0.0, 0.3, 0.9, 0.2],   # bank  (a bit of finance, lots of nature)
    [1.0, 0.0, 0.0, 0.0],   # of    (function word)
    [1.0, 0.0, 0.0, 0.0],   # the   (function word)
    [0.0, 0.0, 0.9, 0.9],   # river (nature + water)
]).unsqueeze(0)  # add batch dim -> (1, 5, 4)

# Use identity projections so Q = K = V = the embeddings themselves.
# (This is the "already trained, no extra transform" case, just to expose the mechanism.)
_, attn_weights = scaled_dot_product_attention(emb, emb, emb)

attn = attn_weights[0]  # (5, 5): row = "querying word", col = "word being looked at"

print("Attention weights (rows query, columns are attended-to):\n")
header = "          " + "".join(f"{w:>8}" for w in words)
print(header)
for i, w in enumerate(words):
    row = "".join(f"{attn[i, j]:8.2f}" for j in range(len(words)))
    print(f"{w:>10}{row}")

# The row for "bank" should put more weight on "river" than on the function words.
bank_i, river_i = words.index("bank"), words.index("river")
print(f"\n'bank' -> 'river' weight: {attn[bank_i, river_i]:.2f}  (highest among other words)")
print(f"'bank' -> 'the'   weight: {attn[bank_i, 0]:.2f}")

# Verify every row of a softmax is a valid probability distribution (sums to 1).
print("\nRow sums (should all be 1.0):", [round(v, 3) for v in attn.sum(dim=-1).tolist()])

plt.figure(figsize=(5, 4))
plt.imshow(attn.numpy(), cmap="viridis")
plt.xticks(range(len(words)), words)
plt.yticks(range(len(words)), words)
plt.xlabel("Looking AT")
plt.ylabel("Querying word")
plt.title("Attention weights")
plt.colorbar()
plt.show()

## Summary

One head of attention is **four lines of tensor math**:

```python
scores  = Q @ K.transpose(-2, -1) / sqrt(d_k)   # who is relevant to whom
weights = softmax(scores, dim=-1)               # normalise to weights summing to 1
output  = weights @ V                           # weighted average of the values
```

...wrapped in three learnable projections (`W_q`, `W_k`, `W_v`) that let the model
*decide* what "relevant" means. Everything else in a Transformer is built around
this core.

Two things are still missing, and they're the subject of **Module 3.2**:

1. One head has to find a single compromise notion of relevance. Real models run
   **many heads in parallel** so different heads can specialise.
2. Right now every word can see *every* word — including the ones that come after it.
   For a model that generates text left to right, that's letting it read the answer.
   The **causal mask** fixes it.

### 🏋️ Try it yourself

1. **Break the scaling.** Delete the `/ sqrt(d_k)` from `scaled_dot_product_attention`
   and re-run the hand-crafted `bank`/`river` demo with the embeddings multiplied by
   `10.0`. What happens to the attention weights? (You should see them collapse toward
   a one-hot "hard max" — this is exactly the saturation Module 1.2 warned about.)
2. **Make a word change its mind.** Edit the `emb` table so `bank` carries a strong
   *finance* component instead of *nature*, and add a `money` token. Confirm that
   `bank` now attends to `money` rather than `river`. You are hand-writing what
   training would otherwise discover.
3. **Prove the residual claim.** The notebook says attention's output has the same
   shape as its input, so `x + attention(x)` is legal. Verify it in one line for the
   `SingleHeadAttention` layer — then remember this in Module 4.1, where it becomes
   the residual connection.

In [ ]:
# Your turn. Task 1 starter: scaling off, inputs scaled up.
from math import sqrt

def unscaled_attention(query, key, value):
    """Deliberately WRONG: no 1/sqrt(d_k)."""
    scores = torch.matmul(query, key.transpose(-2, -1))       # <- the missing divide
    return torch.softmax(scores, dim=-1)

big = emb * 10.0
print("With scaling   :", scaled_dot_product_attention(big, big, big)[1][0, 1].round(decimals=3).tolist())
print("Without scaling:", unscaled_attention(big, big, big)[0, 1].round(decimals=3).tolist())
print("\n(Row shown is the 'bank' query. Without the divide it saturates to a hard max.)")

# Task 3 starter:
x = torch.randn(1, 5, 128)
out, _ = single_head(x)
print("\nresidual works:", (x + out).shape == x.shape)